In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 0. 绘图规范与中文字体设置 (Windows 兼容)
# ==========================================
plt.rcParams['font.sans-serif'] = ['SimHei']  # 设置黑体
plt.rcParams['axes.unicode_minus'] = False    # 修复负号显示
sns.set_context("paper", font_scale=1.2)
sns.set_style("ticks", {"font.sans-serif": ["SimHei"]})

RESULT_DIR = '../results/'
os.makedirs(RESULT_DIR, exist_ok=True)

groups = ['G1', 'G2', 'G3', 'G3-M']

# ==========================================
# 1. 图 5-3 & 5-4：RMSE 和 MAE 分组柱状图
# ==========================================
print(">>> 正在生成 图 5-3 和 5-4: 评估指标分组柱状图...")

lstm_metrics, attn_metrics = [], []
for g in groups:
    # 读取落盘的预测结果自动计算精准指标
    df_lstm = pd.read_csv(os.path.join(RESULT_DIR, f'lstm_pred_{g}.csv'))
    df_attn = pd.read_csv(os.path.join(RESULT_DIR, f'attn_pred_{g}.csv'))
    
    lstm_metrics.append({
        'Group': g,
        'RMSE': np.sqrt(mean_squared_error(df_lstm['true_volatility'], df_lstm['pred_volatility'])),
        'MAE': mean_absolute_error(df_lstm['true_volatility'], df_lstm['pred_volatility'])
    })
    attn_metrics.append({
        'Group': g,
        'RMSE': np.sqrt(mean_squared_error(df_attn['true_volatility'], df_attn['pred_volatility'])),
        'MAE': mean_absolute_error(df_attn['true_volatility'], df_attn['pred_volatility'])
    })

df_lstm_res = pd.DataFrame(lstm_metrics)
df_attn_res = pd.DataFrame(attn_metrics)

x = np.arange(len(groups))
width = 0.35

# 绘制 图 5-3: RMSE 柱状图
fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
rects1 = ax.bar(x - width/2, df_lstm_res['RMSE'], width, label='基线模型 (LSTM)', color='#a6cee3', edgecolor='black')
rects2 = ax.bar(x + width/2, df_attn_res['RMSE'], width, label='主模型 (Bi-LSTM+Attn)', color='#1f78b4', edgecolor='black')

ax.set_ylabel('均方根误差 (RMSE)', fontsize=12)
ax.set_xlabel('特征分组 (Feature Groups)', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, '5_3_rmse_comparison.png'), format='png', bbox_inches='tight')
plt.close()

# 绘制 图 5-4: MAE 柱状图
fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
rects1 = ax.bar(x - width/2, df_lstm_res['MAE'], width, label='基线模型 (LSTM)', color='#b2df8a', edgecolor='black')
rects2 = ax.bar(x + width/2, df_attn_res['MAE'], width, label='主模型 (Bi-LSTM+Attn)', color='#33a02c', edgecolor='black')

ax.set_ylabel('平均绝对误差 (MAE)', fontsize=12)
ax.set_xlabel('特征分组 (Feature Groups)', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, '5_4_mae_comparison.png'), format='png', bbox_inches='tight')
plt.close()

# ==========================================
# 2. 图 5-5：测试集波动率预测时间序列图 (主图 + 极值放大子图)
# ==========================================
print(">>> 正在生成 图 5-5: 波动率预测与极值放大时间序列图...")

df_lstm_g3 = pd.read_csv(os.path.join(RESULT_DIR, 'lstm_pred_G3.csv'))
df_attn_g3 = pd.read_csv(os.path.join(RESULT_DIR, 'attn_pred_G3.csv'))

df_lstm_g3['timestamp'] = pd.to_datetime(df_lstm_g3['timestamp'])
df_attn_g3['timestamp'] = pd.to_datetime(df_attn_g3['timestamp'])

# 自动寻找波动率极值点所在的索引，切出前后15天的窗口用于子图放大
peak_idx = df_attn_g3['true_volatility'].idxmax()
zoom_start = max(0, peak_idx - 15)
zoom_end = min(len(df_attn_g3), peak_idx + 15)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={'height_ratios': [2, 1]}, dpi=300)

# 上半部分：全局走势
ax1.plot(df_attn_g3['timestamp'], df_attn_g3['true_volatility'], label='真实波动率 (True Volatility)', color='black', linewidth=1.5, alpha=0.8)
ax1.plot(df_lstm_g3['timestamp'], df_lstm_g3['pred_volatility'], label='LSTM 预测', color='#1f77b4', linewidth=1.2, linestyle='--')
ax1.plot(df_attn_g3['timestamp'], df_attn_g3['pred_volatility'], label='Bi-LSTM+Attn 预测', color='#d62728', linewidth=1.5)

ax1.set_ylabel('次日对数极值波动率 ($\sigma_{t+1}$)', fontsize=12)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.legend(loc='upper right')
ax1.grid(True, linestyle=':', alpha=0.7)

# 下半部分：局部极值放大
zoom_df_attn = df_attn_g3.iloc[zoom_start:zoom_end]
zoom_df_lstm = df_lstm_g3.iloc[zoom_start:zoom_end]

ax2.plot(zoom_df_attn['timestamp'], zoom_df_attn['true_volatility'], color='black', linewidth=2, marker='o', markersize=4)
ax2.plot(zoom_df_lstm['timestamp'], zoom_df_lstm['pred_volatility'], color='#1f77b4', linewidth=2, linestyle='--', marker='s', markersize=4)
ax2.plot(zoom_df_attn['timestamp'], zoom_df_attn['pred_volatility'], color='#d62728', linewidth=2, marker='^', markersize=4)

ax2.set_xlabel('日期 (Date)', fontsize=12)
ax2.set_ylabel('局部波动率放大', fontsize=12)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax2.grid(True, linestyle=':', alpha=0.7)

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, '5_5_volatility_prediction.png'), format='png', bbox_inches='tight')
plt.close()

# ==========================================
# 3. 图 5-6：隐藏层维度敏感性分析折线图
# ==========================================
print(">>> 正在生成 图 5-6: 网格搜索隐藏层敏感性分析图...")
gs_file = os.path.join(RESULT_DIR, 'grid_search_results_baseline.csv')
if os.path.exists(gs_file):
    df_gs = pd.read_csv(gs_file)
    # 取每个特征组下，不同 hidden_size 对应的最小 RMSE
    df_sens = df_gs.groupby(['feature_group', 'hidden_size'])['RMSE'].min().unstack('feature_group')
    
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    colors = {'G1': '#7f7f7f', 'G2': '#17becf', 'G3': '#d62728', 'G3-M': '#ff7f0e'}
    markers = {'G1': 'o', 'G2': 's', 'G3': '^', 'G3-M': 'D'}
    
    for g in df_sens.columns:
        if g in colors:
            ax.plot(df_sens.index, df_sens[g], label=f'{g} 最优 RMSE', 
                    color=colors[g], marker=markers[g], linewidth=2, markersize=8)
            
    ax.set_xlabel('隐藏层维度 (Hidden Size)', fontsize=12)
    ax.set_ylabel('验证集最优 RMSE', fontsize=12)
    # 强制让 X 轴只显示我们有的维度 (如 64, 128)
    ax.set_xticks(df_sens.index.unique())
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, '5_6_sensitivity_hiddensize.png'), format='png', bbox_inches='tight')
    plt.close()
else:
    print(f"    [跳过] 找不到文件 {gs_file}")

# ==========================================
# 4. 图 5-7 & 5-8：Attention 权重可视化
# ==========================================
print(">>> 正在生成 图 5-7 和 5-8: Attention 权重热力图与柱状图...")
attn_weights = np.load(os.path.join(RESULT_DIR, 'attn_weights_G3.npy'))

# 时间步标签: t-7 到 t-1
time_steps = [f't-{i}' for i in range(7, 0, -1)]

# 图 5-7: 典型波动行情 Attention 热力图 (取刚才找到的极大值附近的 10 天)
start_heatmap = max(0, peak_idx - 5)
end_heatmap = min(len(attn_weights), peak_idx + 5)
sample_attn = attn_weights[start_heatmap:end_heatmap]
sample_dates = df_attn_g3['timestamp'].dt.strftime('%m-%d').iloc[start_heatmap:end_heatmap].values

fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
sns.heatmap(sample_attn, annot=True, fmt=".4f", cmap="YlOrRd", 
            xticklabels=time_steps, yticklabels=sample_dates, ax=ax, cbar_kws={'label': '注意力权重 (Attention Weight)'})
ax.set_xlabel('过去时间步 (Lag Days)', fontsize=12)
ax.set_ylabel('目标预测日期 (Date)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, '5_7_attention_heatmap.png'), format='png', bbox_inches='tight')
plt.close()

# 图 5-8: 测试集全局平均 Attention 权重柱状图
avg_attn = np.mean(attn_weights, axis=0)

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
bars = ax.bar(time_steps, avg_attn, color='#ff9896', edgecolor='#d62728', linewidth=1.5)

# 在柱子上添加具体数值
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.002, round(yval, 4), ha='center', va='bottom', fontsize=10)

ax.set_xlabel('过去时间步 (Lag Days)', fontsize=12)
ax.set_ylabel('平均注意力权重 (Average Attention Weight)', fontsize=12)
# 强制Y轴从0开始，并留出少许顶部空间显示数字
ax.set_ylim(0, max(avg_attn) * 1.15)
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, '5_8_attention_avg_bar.png'), format='png', bbox_inches='tight')
plt.close()

print("\n>>> 全部图表绘制完毕！请前往 ../results/ 目录查看。")

>>> 正在生成 图 5-3 和 5-4: 评估指标分组柱状图...
>>> 正在生成 图 5-5: 波动率预测与极值放大时间序列图...
>>> 正在生成 图 5-6: 网格搜索隐藏层敏感性分析图...
>>> 正在生成 图 5-7 和 5-8: Attention 权重热力图与柱状图...

>>> 全部图表绘制完毕！请前往 ../results/ 目录查看。
